<a href="https://colab.research.google.com/github/Claudiopj88/Atividades_ANN/blob/main/Atividade_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("avnishnish/mnist-original")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'mnist-original' dataset.
Path to dataset files: /kaggle/input/mnist-original


In [29]:
import scipy.io
import os
import numpy as np
from sklearn.model_selection import train_test_split

files = os.listdir(path)
mat_file = [f for f in files if f.endswith('.mat')][0]
full_path = os.path.join(path, mat_file)

data = scipy.io.loadmat(full_path)



In [35]:
X = data['data'].T
y = data['label'].flatten()

In [36]:
X = X.reshape(-1, 28, 28,1)
X.shape

X = X.astype('float32') / 255.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X.shape


(70000, 28, 28, 1)

In [37]:
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np

def KerasCNN2D(input_shape, num_classes):
  inputs = keras.Input(shape=input_shape)
  x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(inputs)
  x = layers.MaxPooling2D(pool_size=2)(x)
  x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
  x = layers.MaxPooling2D(pool_size=2)(x)
  x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
  x = layers.Flatten()(x)
  outputs = layers.Dense(num_classes, activation="softmax")(x)
  return keras.Model(inputs=inputs, outputs=outputs)

model = KerasCNN2D((28, 28, 1), 10)
model.summary()



Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_36 (Conv2D)              │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_37 (Conv2D)              │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_25 (MaxPooling2D) │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_38 (Conv2D)              │ (None, 3, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_12 (Flatten)            │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 10)             │        11,530 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 104,202 (407.04 KB)

 Trainable params: 104,202 (407.04 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
class KerasWrappedNN(BaseEstimator, ClassifierMixin):
  def __init__(self, epochs=5, batch_size=128, model_fabric=KerasCNN2D):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    y_hot = keras.utils.to_categorical(ids, len(self.labels))
    self.model = self.model_fabric(X.shape[1:], num_classes=len(np.unique(y)))
    self.model.compile(
        optimizer="rmsprop",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    self.model.fit(
        X, y_hot,
        epochs=self.epochs,
        batch_size=self.batch_size
    )
    return self

  def predict(self, X):
    y_pred = self.model.predict(X)
    return self.labels[np.argmax(y_pred, axis=1)]

In [39]:
model = KerasWrappedNN()
model.fit( X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 30s 68ms/step - accuracy: 0.9242 - loss: 0.2457
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.9821 - loss: 0.0559
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.9881 - loss: 0.0376
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 39s 63ms/step - accuracy: 0.9906 - loss: 0.0279
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.9931 - loss: 0.0212
438/438 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step


0.9885714285714285